En este notebook probaremos los distintos modelos y liberias de boosting que se adapnten a nuestro problema ( precision de clases alta y balanceada, y velocidad)

La idea es crear atributos mediante la extraccion de caracteristias que sean entendibles para nuetsros modelos o algoritmos de boosting, ya que sino, ellos no entienden de series temporales y no significa nada el valor de un canal en el instante 1 o en el 56, cuando es algo de vital importancia.

Las carcateristicas seran estadisticas de los canales por muestra, por ejemeplo, la media, la desviacion tipica, la diferencia entre el valor maximo y el minimo, la frecuencia, etc. Por lo que si extraemos 10 estadisticas, nos saldran 40 atributos ( 10 por cada 1 de los 4 canales ).

Al parecer esta cantidad de atributo es mas q manejable y no relantizara la prediccion.

In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from competition_api import load_dataset

Empezampos imputando los nans

In [2]:
def imputar_nans(serie):
    """
    Rellena los NaNs de una serie (4, 256) usando interpolación lineal a lo largo del eje temporal.
    Si los extremos son NaN, hace forward o backward fill.
    """
    # Convertimos a DataFrame transpuesto (256 filas, 4 columnas) para usar el interpolate hiper-optimizado de Pandas
    df = pd.DataFrame(serie.T)
    # Interpolar linealmente los huecos internos, luego rellenar extremos si quedaron NaNs
    df.interpolate(method='linear', limit_direction='both', inplace=True)
    return df.values.T # Devolvemos a la forma (4, 256)

print("Cargando y limpiando datos...")
ds = load_dataset("data/train_100k.npz")
X_raw = ds.series
y = ds.labels

# Imputamos todo el dataset de golpe (puede tardar un par de minutos, esto solo se hace en entrenamiento)
X_clean = np.array([imputar_nans(muestra) for muestra in X_raw])
print(f"Datos limpios. Total NaNs restantes: {np.isnan(X_clean).sum()}")

Cargando y limpiando datos...
Datos limpios. Total NaNs restantes: 0


Con la extraccion de carcateristicas paree no ser sficiente, el modelo se sobreajusta y ante un posible cambio de la escala o de la escala o de la forma d elos datos el modelo se vuelve ciego. Estandarizar es ua opcion, pero nos quita estadisticas buenas e interesantes como la media o la desviacion tipica. Por eso, suavizmaos los atos y creamos ruido pra obligar al modelo a detectar patrons reales y aprenderse e meoria los datos.

In [3]:
from scipy.ndimage import uniform_filter1d

def suavizar_y_aumentar(X_limpio, y_original, num_copias=1):
    """
    1. Aplica media móvil para quitar ruido eléctrico (suavizado).
    2. Genera copias mutadas con Jitter, Shift y Scale.
    """
    X_aumentado = []
    y_aumentado = []
    
    # 1. Añadir los datos originales, pero pasados por el filtro de suavizado
    for serie, label in zip(X_limpio, y_original):
        serie_suave = uniform_filter1d(serie, size=3, axis=1)
        X_aumentado.append(serie_suave)
        y_aumentado.append(label)
        
    # 2. Añadir las copias mutadas (Data Augmentation)
    for _ in range(num_copias):
        for serie, label in zip(X_limpio, y_original):
            serie_suave = uniform_filter1d(serie, size=3, axis=1)
            
            ruido = np.random.normal(loc=0.0, scale=0.1, size=serie.shape)
            shift = np.random.uniform(-1.5, 1.5, size=(4, 1))
            scale = np.random.uniform(0.8, 1.2, size=(4, 1))
            
            serie_mutada = (serie_suave + shift + ruido) * scale
            X_aumentado.append(serie_mutada)
            y_aumentado.append(label)
            
    return np.array(X_aumentado), np.array(y_aumentado)

Creamos la funcion de extraccion de crcateristicas, las cuales hemos elegido en base a la sgraficas de los resultados en el analisi de exploracion de lo datos.

In [4]:
EJE_X = np.arange(256)

def extraer_caracteristicas_pro(serie_limpia):
    """
    Extrae las 42 características que nos dieron el techo del 78%.
    """
    features = []
    for canal in serie_limpia:
        mean = np.mean(canal)
        std = np.std(canal)
        std_safe = std if std > 1e-8 else 1e-8 
        
        features.append(mean)
        features.append(np.max(canal) - np.min(canal))
        features.append(std)
        
        features.append(np.mean(((canal - mean) / std_safe) ** 3))
        features.append(np.mean(((canal - mean) / std_safe) ** 4) - 3)
        
        diffs = np.diff(canal)
        features.append(np.max(np.abs(diffs)))
        features.append(np.var(diffs))
        
        canal_centrado = canal - mean
        features.append(np.sum(np.diff(np.sign(canal_centrado)) != 0))
        
        coef_a = np.polyfit(EJE_X, canal, 2)[0]
        features.append(coef_a)
        
    matriz_corr = np.corrcoef(serie_limpia)
    correlaciones = matriz_corr[np.triu_indices(4, k=1)]
    features.extend(correlaciones.tolist())
    
    return np.nan_to_num(np.array(features), nan=0.0) 

In [5]:
print("1. Cargando y limpiando Train...")
ds_train = load_dataset("data/train_100k.npz")
X_train_raw = ds_train.series
y_train_raw = ds_train.labels

X_train_limpio = np.array([imputar_nans(m) for m in X_train_raw])

print("2. Aplicando Data Augmentation y Suavizado...")
# Generamos 1 copia mutada por cada original. Pasamos de 100k a 200k muestras.
X_train_aug, y_train_aug = suavizar_y_aumentar(X_train_limpio, y_train_raw, num_copias=1)
print(f"   -> Muestras de entrenamiento generadas: {len(X_train_aug)}")

print("3. Extrayendo las 42 características...")
X_train_features = np.array([extraer_caracteristicas_pro(m) for m in X_train_aug])
print(f"   -> Forma final del dataset: {X_train_features.shape}")

X_train, X_val, y_train, y_val = train_test_split(
    X_train_features, y_train_aug, test_size=0.2, random_state=42, stratify=y_train_aug
)

1. Cargando y limpiando Train...
2. Aplicando Data Augmentation y Suavizado...
   -> Muestras de entrenamiento generadas: 200000
3. Extrayendo las 42 características...
   -> Forma final del dataset: (200000, 42)


Ahora pasamos a una primera inspeccion de que tal les va a los modelos, para ir descartando posibilidades. Sin embargo, hay que tener en cueta que los hiperparametros no estan optimizados todavia, detalle importnte.

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import time

print("4. Entrenando modelos con datos aumentados (esto tardará un poco más al haber 200k muestras)...")

modelos = {
    "HistGradientBoosting (Sklearn)": HistGradientBoostingClassifier(random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
    "XGBoost": XGBClassifier(random_state=42, n_jobs=-1, eval_metric='mlogloss')
}

resultados_val = {}

for nombre, modelo in modelos.items():
    print(f"   Entrenando {nombre}...")
    
    t0 = time.time()
    modelo.fit(X_train, y_train)
    t_train = time.time() - t0
    
    preds_val = modelo.predict(X_val)
    acc_val = accuracy_score(y_val, preds_val)
    
    # Inferencia de 1 muestra para ver si cumplen el límite de 0.5s
    muestra_suelta = X_val[0].reshape(1, -1)
    t0_inf = time.perf_counter()
    _ = modelo.predict(muestra_suelta)
    t_inf = time.perf_counter() - t0_inf
    
    resultados_val[nombre] = {
        "Accuracy_Validation": acc_val, 
        "Inferencia_1_muestra(s)": t_inf, 
        "Tiempo_entrenamiento(s)": t_train
    }

4. Entrenando modelos con datos aumentados (esto tardará un poco más al haber 200k muestras)...
   Entrenando HistGradientBoosting (Sklearn)...
   Entrenando LightGBM...
   Entrenando XGBoost...


Imprimimos las metricas.

In [8]:
import pandas as pd
from sklearn.metrics import classification_report

print("5. EVALUACIÓN EN EL TEST DEMO...")
ds_test = load_dataset("data/test_demo.npz")

print("   -> Limpiando NaNs del test...")
X_test_clean = np.array([imputar_nans(m) for m in ds_test.series])

print("   -> Suavizando el test (OJO: Sin mutaciones, solo el filtro)...")
X_test_suave = np.array([uniform_filter1d(m, size=3, axis=1) for m in X_test_clean])

print("   -> Extrayendo características y prediciendo...")
X_test_features = np.array([extraer_caracteristicas_pro(m) for m in X_test_suave])

# Evaluar todos los modelos en el Test
filas_tabla = []
y_test_real = ds_test.labels

for nombre, modelo in modelos.items():
    preds_test = modelo.predict(X_test_features)
    acc_test = accuracy_score(y_test_real, preds_test)
    
    fila = resultados_val[nombre].copy()
    fila["Accuracy_Test_Demo"] = acc_test
    filas_tabla.append(fila)

# Construir tabla final comparativa
df_final = pd.DataFrame(filas_tabla, index=modelos.keys())
columnas_ordenadas = ["Accuracy_Validation", "Accuracy_Test_Demo", "Inferencia_1_muestra(s)", "Tiempo_entrenamiento(s)"]
df_final = df_final[columnas_ordenadas]

print("\n--- RESULTADOS COMPLETOS (TRAIN AUMENTADO vs TEST DEMO) ---")
print(df_final.sort_values(by="Accuracy_Test_Demo", ascending=False))

# Analizar al detalle el que mejor haya rendido en el Test
mejor_modelo_nombre = df_final["Accuracy_Test_Demo"].idxmax()
mejor_modelo = modelos[mejor_modelo_nombre]

print(f"\n--- REPORTE POR CLASE DEL GANADOR EN TEST ({mejor_modelo_nombre}) ---")
preds_ganador = mejor_modelo.predict(X_test_features)
print(classification_report(y_test_real, preds_ganador))

5. EVALUACIÓN EN EL TEST DEMO...
   -> Limpiando NaNs del test...
   -> Suavizando el test (OJO: Sin mutaciones, solo el filtro)...
   -> Extrayendo características y prediciendo...

--- RESULTADOS COMPLETOS (TRAIN AUMENTADO vs TEST DEMO) ---
                                Accuracy_Validation  Accuracy_Test_Demo  \
XGBoost                                    0.749375              0.4650   
HistGradientBoosting (Sklearn)             0.736150              0.4550   
LightGBM                                   0.734025              0.4475   

                                Inferencia_1_muestra(s)  \
XGBoost                                        0.000422   
HistGradientBoosting (Sklearn)                 0.009217   
LightGBM                                       0.000911   

                                Tiempo_entrenamiento(s)  
XGBoost                                        7.611118  
HistGradientBoosting (Sklearn)                 3.584477  
LightGBM                                     

A la vista d elos resultados, el ganador es xgboost, pero opimizaremos tmb lo hiperpaametros de lightgbm ya que este es el mas rapido, y es algo que tmb necesitamos.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
import time

print("Iniciando optimización de hiperparámetros doble (XGBoost y LightGBM)...")
print("Esto va a tardar. Ve a por un café.\n")

# 1. Definir los modelos base y sus espacios de búsqueda de hiperparámetros
configuraciones = {
    "LightGBM": {
        "modelo": LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
        "params": {
            'num_leaves': randint(15, 60),           # Evitar árboles muy complejos
            'max_depth': randint(3, 10),             # Poca profundidad = menos sobreajuste
            'learning_rate': uniform(0.01, 0.15),
            'n_estimators': randint(100, 300),
            'subsample': uniform(0.5, 0.4),          # Usar % aleatorio de muestras
            'colsample_bytree': uniform(0.5, 0.4),   # Usar % aleatorio de variables
            'min_child_samples': randint(20, 100)    # Exigir datos en las hojas
        }
    },
    "XGBoost": {
        "modelo": XGBClassifier(random_state=42, n_jobs=-1, eval_metric='mlogloss'),
        "params": {
            'max_depth': randint(3, 8),              
            'learning_rate': uniform(0.01, 0.15),     
            'n_estimators': randint(100, 300),       
            'subsample': uniform(0.5, 0.4),          
            'colsample_bytree': uniform(0.5, 0.4),   
            'min_child_weight': randint(2, 10)       
        }
    }
}

mejores_modelos = {}
resultados_optimos = {}

# 2. Ejecutar la búsqueda para cada modelo
# Usamos n_iter=10 para no eternizar la búsqueda (10 combinaciones aleatorias)
for nombre, config in configuraciones.items():
    print(f"--- Optimizando {nombre} ---")
    
    random_search = RandomizedSearchCV(
        estimator=config["modelo"],
        param_distributions=config["params"],
        n_iter=10, 
        cv=3, 
        scoring='accuracy',
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    
    t0 = time.time()
    random_search.fit(X_train, y_train)
    t_search = time.time() - t0
    
    print(f"Búsqueda {nombre} completada en {t_search / 60:.1f} minutos.")
    print(f"Mejor Accuracy en CV (Train Aumentado): {random_search.best_score_:.4f}")
    
    # Guardar el mejor modelo
    mejor_modelo = random_search.best_estimator_
    mejores_modelos[nombre] = mejor_modelo
    
    # Probarlo inmediatamente contra el TEST DEMO
    preds_test = mejor_modelo.predict(X_test_features)
    acc_test = accuracy_score(y_test_real, preds_test)
    
    resultados_optimos[nombre] = {
        "Mejores_Params": random_search.best_params_,
        "Accuracy_Test": acc_test
    }
    
    print(f"-> Accuracy del {nombre} OPTIMIZADO en TEST DEMO: {acc_test:.5f}\n")

# 3. Resumen final
print("--- RESUMEN FINAL TRAS OPTIMIZACIÓN ---")
for nombre, res in resultados_optimos.items():
    print(f"{nombre}: {res['Accuracy_Test']:.5f}")